In [1]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from util import *
from distorted import step1, step2, step3, fix1, evaluation, prompts
load_dotenv()
MODEL = "gpt-5.1"
llm = llm_call(model_version=MODEL, api_key=os.getenv("API_KEY"))

LOG_NAME = "credit_seed52_ratio0.3_distorted_withLabel"
df, _ = build_event_jsons(log_name=f"./dataset/{LOG_NAME}.csv", chunk_cases=1)
llm_repetition = 10

fix_repetition = 3
current_df = df.copy() 

for iteration in range(fix_repetition):
    print(f"\n{'='*30}")
    print(f">>> ITERATION {iteration + 1} START")
    print(f"{'='*30}")
    activity_list = current_df['activity'].unique().tolist()
    activity_list_json = json.dumps(activity_list, indent=4, ensure_ascii=False)
    
    res_s1 = step1.run_step1(llm, MODEL, llm_repetition, activity_list_json, 
                             prompts.SYSTEM_PROMPT_DISTORTED_STEP1, 
                             prompts.USER_PROMPT_DISTORTED_STEP1)
    if not res_s1:
        print(f">>> No more distorted candidates found in iteration {iteration + 1}. Breaking loop.")
        break

    act_freq_dict = current_df['activity'].value_counts().to_dict()
    act_freq_dict_json = json.dumps(act_freq_dict, indent=4, ensure_ascii=False)

    res_s2 = step2.run_step2(llm, MODEL, llm_repetition, res_s1, 
                             current_df, 
                             prompts.SYSTEM_PROMPT_DISTORTED_STEP2, 
                             prompts.USER_PROMPT_DISTORTED_STEP2)

    res_s3 = step3.run_step3(llm, MODEL, llm_repetition, polluted_data, # res_s2, 
                             prompts.SYSTEM_PROMPT_DISTORTED_STEP3, 
                             prompts.USER_PROMPT_DISTORTED_STEP3)

    
    print(f"\n>>> Cluster Preview (Total: {len(res_s2)} groups)")
    for i, (clean, variants) in enumerate(res_s2.items()):
        if i >= 3: break  
        print(f"  - {clean}: {variants[:3]} ... (+{len(variants)-3} more)")

    current_df = fix1.run_fix1(current_df, res_s2)

    print("\n>>> EVALUATION")
    eval_summary = evaluation.run_evaluation(current_df, df)

    print(f">>> Iteration {iteration + 1} complete.")





>>> ITERATION 1 START
>>> Running Step 1  with 10 repetitions...
Candidates found (10 total): ['Check for completeness', 'Deliver card', 'Make decision', 'Notify accept', 'Perform checks'] ...
>>> Running Step 2  with 10 repetitions...


NameError: name 'polluted_data' is not defined

In [29]:
res_s1

{'found': True,
 'original_activity': ['Check for completeness',
  'Deliver card',
  'Make decision',
  'Notify accept',
  'Perform checks',
  'Request info',
  'info received',
  'notify reject',
  'review request received',
  'time out']}

In [2]:
LOG_NAME = "pub_seed52_ratio0.3_distorted_withLabel"
df, _ = build_event_jsons(log_name=f"./dataset/{LOG_NAME}.csv", chunk_cases=1)
llm_repetition = 10

fix_repetition = 3
current_df = df.copy() 

for iteration in range(fix_repetition):
    print(f"\n{'='*30}")
    print(f">>> ITERATION {iteration + 1} START")
    print(f"{'='*30}")
    activity_list = current_df['activity'].unique().tolist()
    activity_list_json = json.dumps(activity_list, indent=4, ensure_ascii=False)
    
    res_s1 = step1.run_step1(llm, MODEL, llm_repetition, activity_list_json, 
                             prompts.SYSTEM_PROMPT_DISTORTED_STEP1, 
                             prompts.USER_PROMPT_DISTORTED_STEP1)

    if not res_s1['original_activity']:
        print(f">>> No more distorted candidates found in iteration {iteration + 1}. Breaking loop.")
        break
    print(f"Candidates found ({len(res_s1['original_activity'])} total): {res_s1['original_activity'][:5]} ...")

    clean_labels = res_s1['original_activity']
    act_freq_dict = current_df['activity'].value_counts().to_dict()
    act_freq_dict_json = json.dumps(act_freq_dict, indent=4, ensure_ascii=False)

    res_s2 = step2.run_step2(llm, MODEL, llm_repetition, clean_labels, 
                             act_freq_dict_json, 
                             prompts.SYSTEM_PROMPT_DISTORTED_STEP2, 
                             prompts.USER_PROMPT_DISTORTED_STEP2)

    print(f"\n>>> Cluster Preview (Total: {len(res_s2)} groups)")
    for i, (clean, variants) in enumerate(res_s2.items()):
        if i >= 3: break  
        print(f"  - {clean}: {variants[:3]} ... (+{len(variants)-3} more)")

    current_df = fix1.run_fix1(current_df, res_s2)

    print("\n>>> EVALUATION")
    eval_summary = evaluation.run_evaluation(current_df, df)

    print(f">>> Iteration {iteration + 1} complete.")





>>> ITERATION 1 START
>>> Running Step 1  with 10 repetitions...
Candidates found (9 total): ['Bring drinks', 'Bring food', 'Deliver to customer', 'Prepare main course', 'Prepare starter'] ...
>>> Running Step 2  with 10 repetitions...

>>> Cluster Preview (Total: 9 groups)
  - Bring drinks: ['Birng drinks', 'BrIng drinKs', 'Brig drinks'] ... (+19 more)
  - Bring food: ['BrJing food', 'BriMng food', 'BriNg fOod'] ... (+21 more)
  - Deliver to customer: ['DElIver tO custOmEr', 'De,iver to customer', 'DeLiver to CUsTomeR'] ... (+21 more)
>>> Starting Activity Correction...
>>> Correction complete. Total rows processed: 56524

>>> EVALUATION

>>> Starting Evaluation...
------------------------------
EVALUATION SUMMARY
------------------------------
eval_status
Correct (TN)    49048
Correct (TP)     5849
Missed (FN)      1627
------------------------------
Recall:    78.24%
Precision: 100.00%
------------------------------

>>> Error Samples (Top 5):
       original_activity             a

In [1]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from util import *
from distorted import step1, step2, fix1, evaluation, prompts
load_dotenv()
MODEL = "gpt-5.1"
llm = llm_call(model_version=MODEL, api_key=os.getenv("API_KEY"))

LOG_NAME = "credit_seed52_ratio0.3_distorted_withLabel"
df, _ = build_event_jsons(log_name=f"./dataset/{LOG_NAME}.csv", chunk_cases=1)
llm_repetition = 5

fix_repetition = 1
current_df = df.copy() 

for iteration in range(fix_repetition):
    print(f"\n{'='*30}")
    print(f">>> ITERATION {iteration + 1} START")
    print(f"{'='*30}")
    activity_list = current_df['activity'].unique().tolist()
    activity_list_json = json.dumps(activity_list, indent=4, ensure_ascii=False)
    
    res_s1 = step1.run_step1(llm, MODEL, llm_repetition, activity_list_json, 
                             prompts.SYSTEM_PROMPT_DISTORTED_STEP1, 
                             prompts.USER_PROMPT_DISTORTED_STEP1)

    if not res_s1['original_activity']:
        print(f">>> No more distorted candidates found in iteration {iteration + 1}. Breaking loop.")
        break
    print(f"Candidates found ({len(res_s1['original_activity'])} total): {res_s1['original_activity'][:5]} ...")

    clean_labels = res_s1['original_activity']
    act_freq_dict = current_df['activity'].value_counts().to_dict()
    act_freq_dict_json = json.dumps(act_freq_dict, indent=4, ensure_ascii=False)

    res_s2 = step2.run_step2(llm, MODEL, llm_repetition, clean_labels, 
                             act_freq_dict_json, 
                             prompts.SYSTEM_PROMPT_DISTORTED_STEP2, 
                             prompts.USER_PROMPT_DISTORTED_STEP2)

    print(f"\n>>> Cluster Preview (Total: {len(res_s2)} groups)")
    for i, (clean, variants) in enumerate(res_s2.items()):
        if i >= 3: break  
        print(f"  - {clean}: {variants[:3]} ... (+{len(variants)-3} more)")



>>> ITERATION 1 START
>>> Running Step 1  with 5 repetitions...
Candidates found (10 total): ['Check for completeness', 'Deliver card', 'Make decision', 'Notify accept', 'Perform checks'] ...
>>> Running Step 2  with 5 repetitions...

>>> Cluster Preview (Total: 10 groups)
  - Check for completeness: ['CHeCk foR comPlEteneSs', 'Ceck for completeness', 'Cehck for completeness'] ... (+22 more)
  - Deliver card: ['DElivEr cArd', 'DEliver card', 'DeLiver card'] ... (+20 more)
  - Make decision: [',ake decision', 'MaRke decision', 'Mak decision'] ... (+22 more)


In [2]:
res_s2

{'Check for completeness': ['CHeCk foR comPlEteneSs',
  'Ceck for completeness',
  'Cehck for completeness',
  'ChEck for cOMpleteNesS',
  'ChEck for completeness',
  'ChecK for coMpleTenesS',
  'Check fOr complEteness',
  'Check fo completeness',
  'Check for cCompleteness',
  'Check for cmpleteness',
  'Check for cojpleteness',
  'Check for comcpleteness',
  'Check for comlpleteness',
  'Check for compelteness',
  'Check for compleeness',
  'Check for completebess',
  'Check for completeneass',
  'Check for completneess',
  'Check for complfteness',
  'Check for complteness',
  'Check for copmleteness',
  'Check for sompleteness',
  'Checkk for completeness',
  'Dheck for completeness',
  'hCeck for completeness'],
 'Deliver card': ['DElivEr cArd',
  'DEliver card',
  'DeLiver card',
  'Deiver card',
  'Delier card',
  'DelivJer card',
  'Deliver acrd',
  'Deliver ard',
  'Deliver caXrd',
  'Deliver carD',
  'Deliver crad',
  'Deliver fard',
  'Deliverc ard',
  'Deliverr card',
  'De

In [14]:
import random
from collections import defaultdict
import copy
polluted_data = copy.deepcopy(res_s2)
all_keys = list(polluted_data.keys())

print("=== 1. 완전 랜덤 데이터 오염(중복 생성) 시작 ===")

pollution_history = []
TOTAL_POLLUTIONS = 10  # 🎯 생성하고 싶은 중복 케이스 개수 지정
for i in range(TOTAL_POLLUTIONS):
    from_key = random.choice(all_keys)
    target_element = random.choice(polluted_data[from_key])
    candidate_keys = [k for k in all_keys if k != from_key]
    num_to_infect = random.randint(1, min(2, len(candidate_keys)))
    to_keys = random.sample(candidate_keys, num_to_infect)
    for tk in to_keys:
        polluted_data[tk].append(target_element)
        pollution_history.append({
            'element': target_element,
            'from': from_key,
            'to': tk
        })
print(f"\n🎲 시스템이 무작위로 생성한 중복 내역 ({len(pollution_history)}개 매핑):")
for idx, hist in enumerate(pollution_history, 1):
    print(f"   {idx}) 단어 '{hist['element']}' (원본: [{hist['from']}]) ➔ [{hist['to']}] 리스트에 주입됨.")
print("\n" + "="*60 + "\n")
print("🔍 2. [전체 Key 대상] 교차 중복 요소 검증 실행\n")
element_to_keys = defaultdict(list)
for key, words in polluted_data.items():
    for word in words:
        element_to_keys[word].append(key)
has_overlap = False
detected_count = 0
for element, keys in element_to_keys.items():
    if len(keys) > 1:
        has_overlap = True
        detected_count += 1
        unique_keys = sorted(list(set(keys)))
        
        print(f"🚨 [검출 {detected_count}] 중복 요소 발견: '{element}'")
        print(f"   ↳ 현재 이 요소가 포함된 Key 목록: {unique_keys}")
        print("-" * 50)

if not has_overlap:
    print("✅ 중복 요소가 없습니다.")
else:
    print(f"🎯 검증 완료: 랜덤으로 생성된 중복 오염이 정확하게 검출되었습니다.")

=== 1. 완전 랜덤 데이터 오염(중복 생성) 시작 ===

🎲 시스템이 무작위로 생성한 중복 내역 (15개 매핑):
   1) 단어 'NOtify AcCept' (원본: [Notify accept]) ➔ [Perform checks] 리스트에 주입됨.
   2) 단어 'review reques received' (원본: [review request received]) ➔ [time out] 리스트에 주입됨.
   3) 단어 'Requestinfo' (원본: [Request info]) ➔ [notify reject] 리스트에 주입됨.
   4) 단어 'timE out' (원본: [time out]) ➔ [info received] 리스트에 주입됨.
   5) 단어 'timE out' (원본: [time out]) ➔ [Make decision] 리스트에 주입됨.
   6) 단어 'nOTify rejEct' (원본: [notify reject]) ➔ [Notify accept] 리스트에 주입됨.
   7) 단어 'info recieved' (원본: [info received]) ➔ [Request info] 리스트에 주입됨.
   8) 단어 'info recieved' (원본: [info received]) ➔ [Check for completeness] 리스트에 주입됨.
   9) 단어 'inf received' (원본: [info received]) ➔ [Request info] 리스트에 주입됨.
   10) 단어 'inf received' (원본: [info received]) ➔ [Perform checks] 리스트에 주입됨.
   11) 단어 'rime out' (원본: [time out]) ➔ [notify reject] 리스트에 주입됨.
   12) 단어 'rime out' (원본: [time out]) ➔ [Notify accept] 리스트에 주입됨.
   13) 단어 'inFo ReceivEd' (원본: [info received]) ➔ [C

Check for completeness 27
Deliver card 24
Make decision 26
Notify accept 24
Perform checks 26
Request info 26
info received 25
notify reject 27
review request received 25
time out 26

Check for completeness 25
Deliver card 23
Make decision 25
Notify accept 22
Perform checks 24
Request info 22
info received 24
notify reject 25
review request received 25
time out 25

256 240


In [21]:
import importlib
import distorted
import distorted.step3, distorted.prompts

importlib.reload(distorted)
importlib.reload(distorted.step3)
importlib.reload(distorted.prompts)


from distorted import step3
res_s3 = step3.run_step3(llm, MODEL, llm_repetition, polluted_data, # res_s2, 
                         prompts.SYSTEM_PROMPT_DISTORTED_STEP3, 
                         prompts.USER_PROMPT_DISTORTED_STEP3)


>>> Running Step 3  with 5 repetitions...
>>> 10 overlapping noise word(s) to adjudicate.
  - 'info recieved' -> info received (5/5 votes)
  - 'inFo ReceivEd' -> info received (5/5 votes)
  - 'Make decisioN' -> Make decision (5/5 votes)
  - 'timE out' -> time out (5/5 votes)
  - 'NOtify AcCept' -> Notify accept (5/5 votes)
  - 'nOTify rejEct' -> notify reject (5/5 votes)
  - 'rime out' -> time out (5/5 votes)
  - 'inf received' -> info received (5/5 votes)
  - 'Requestinfo' -> None (5/5 votes)
  - 'review reques received' -> review request received (5/5 votes)
>>> Resolution applied: removed 16 conflicting entr(ies).


In [22]:
element_to_keys = defaultdict(list)
for key, words in res_s3.items():
    for word in words:
        element_to_keys[word].append(key)
has_overlap = False
detected_count = 0
for element, keys in element_to_keys.items():
    if len(keys) > 1:
        has_overlap = True
        detected_count += 1
        unique_keys = sorted(list(set(keys)))
        
        print(f"🚨 [검출 {detected_count}] 중복 요소 발견: '{element}'")
        print(f"   ↳ 현재 이 요소가 포함된 Key 목록: {unique_keys}")
        print("-" * 50)

if not has_overlap:
    print("✅ 중복 요소가 없습니다.")
else:
    print(f"🎯 검증 완료: 랜덤으로 생성된 중복 오염이 정확하게 검출되었습니다.")

✅ 중복 요소가 없습니다.


In [ ]:
sum_pol1,sum_pol2 = 0,0
for k,v in polluted_data.items():
    print(k,len(v))
    sum_pol1+=len(v)
    
print()
for k,v in res_s3.items():
    print(k,len(v))
    sum_pol2+=len(v)

print()
print(sum_pol1, sum_pol2)

In [28]:
res_s1

{'found': True,
 'original_activity': ['Check for completeness',
  'Deliver card',
  'Make decision',
  'Notify accept',
  'Perform checks',
  'Request info',
  'info received',
  'notify reject',
  'review request received',
  'time out']}